# Round 3 benchmark on Google Colab: GNNs (GCN, GIN, GAT) + tuned XGBoost

1. **Runtime -> Change runtime type -> T4 GPU** (the GNNs train on the GPU).
2. Put `wl_round3_bundle.zip` in the root of your Google Drive (*My Drive*), or upload it when cell 1 asks.
3. **Runtime -> Run all.** Keep this tab open. Everything is saved to Google Drive as it finishes.
4. **If Colab disconnects:** reconnect and Run all again. Finished runs are skipped automatically.
5. At the end, `round3_results.zip` is in your Google Drive. Download it and give it to Claude.

In [ ]:
# 1. Google Drive + project files (extracted once; later runs reuse them and their saved results)
import glob, os, subprocess, zipfile
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/wl_round3_work'
SUITE = WORK + '/wl_round3/New_Models_Baselines/03_Benchmark_Suite'
if not os.path.exists(SUITE):
    os.makedirs(WORK, exist_ok=True)
    zips = glob.glob('/content/drive/MyDrive/wl_round3_bundle.zip') + glob.glob('/content/wl_round3_bundle.zip')
    if not zips:
        from google.colab import files
        print('Upload wl_round3_bundle.zip')
        files.upload()
        zips = ['/content/wl_round3_bundle.zip']
    zipfile.ZipFile(zips[0]).extractall(WORK)
    print('extracted', zips[0])
else:
    print('resuming in', SUITE)
print(subprocess.run('nproc; free -g; nvidia-smi -L', shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 2. Libraries (torch is already installed on Colab)
!pip -q install lightgbm xgboost scikit-image opencv-python-headless 2>&1 | tail -1
!python -c "import torch, lightgbm, xgboost, sklearn; print('torch', torch.__version__, 'GPU', torch.cuda.is_available(), '| lightgbm', lightgbm.__version__, '| xgboost', xgboost.__version__, '| sklearn', sklearn.__version__)"

In [ ]:
# 3. All round-3 experiments (several hours). Prints 'wave 1 done', 'wave 2 done' and ok/FAILED per log.
#    Live progress: open the Files panel -> drive/MyDrive/wl_round3_work/.../03_Benchmark_Suite/logs
!cd "$SUITE" && bash run_round3.sh

In [ ]:
# 4. Results + logs + machine info -> My Drive/round3_results.zip
!cd "$SUITE" && (nproc; lscpu | grep 'Model name'; free -g; nvidia-smi -L) > logs/machine.txt 2>&1
!cd "$SUITE" && rm -f /content/drive/MyDrive/round3_results.zip && zip -qr /content/drive/MyDrive/round3_results.zip results logs
print(os.path.getsize('/content/drive/MyDrive/round3_results.zip') // 1024, 'KB -> My Drive/round3_results.zip')